# Invoice Region Detection and Business Parameter Extraction Using CNN, SSD, IoU, OCR, and Streamlit

**Member:** Damir
**Role:** OCR, Parameter Extraction, Terms & Conditions Extraction Lead

**Objective:** Run OCR on Jordan's detected region crops, check for required reference parameters, and extract payment terms / due dates / terms & conditions signals.

**Inputs expected:**
- `outputs/predictions/region_predictions.csv` (Jordan)
- `outputs/predictions/stamp_signature_predictions.csv` (Diana)
- `config/required_fields_config.json`

**Outputs generated:**
- `outputs/predictions/ocr_outputs.csv`
- `outputs/predictions/parameter_presence_results.csv`
- `outputs/predictions/terms_extraction_results.csv`
- `outputs/metrics/ocr_parameter_metrics.json`

> Run this notebook top-to-bottom in Google Colab, or locally with the repo's virtualenv.
> Paths are resolved via `src/config.py` (pathlib-based) — never hardcode absolute local paths.


In [ ]:
# --- Google Colab setup cell ---
# If running in Colab: clone the repo (or mount Drive if you cloned there already) and
# install dependencies. Safe to skip locally if the repo is already on disk with deps installed.

import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/<your-org>/invoice-image-processing.git"  # TODO: set this
    REPO_DIR = "/content/invoice-image-processing"

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    os.system("pip install -q -r requirements.txt")

    # Kaggle API credentials (upload kaggle.json when prompted) -- see dataset_sources.md
    from google.colab import files
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        print("Upload your kaggle.json (Kaggle -> Account -> Create New API Token):")
        uploaded = files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        for fname in uploaded:
            os.replace(fname, "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

    print("Colab environment ready. Working directory:", os.getcwd())
else:
    print("Not running in Colab -- assuming local repo checkout with requirements installed.")


In [ ]:
# --- Dataset path setup cell ---
# All paths go through src.config.PATHS (pathlib-based, no hardcoded absolute paths).

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "requirements.txt").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import PATHS, load_label_schema, load_required_fields

print("Repo root:", PATHS.repo_root)
print("Raw data dir:", PATHS.raw_dir)
print("Outputs dir:", PATHS.outputs_dir)

# If raw data isn't present yet, download it (see dataset_sources.md for kaggle.json setup):
#   python scripts/download_datasets.py --dataset all


In [ ]:
# --- Imports cell ---
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.image_preprocessing import preprocess_pipeline, to_grayscale, resize_image, denoise_image, threshold_image, deskew_image
from src.visualization import draw_boxes, show_image_grid
from src.annotation_utils import load_annotations, boxes_for_image
from src.iou import compute_iou, precision_recall_iou, evaluate_predictions_df


## 1. Load Jordan's region predictions and Diana's stamp/signature predictions

In [ ]:
region_predictions_path = PATHS.predictions_dir / "region_predictions.csv"
stamp_sig_predictions_path = PATHS.predictions_dir / "stamp_signature_predictions.csv"

region_predictions = pd.read_csv(region_predictions_path) if region_predictions_path.exists() else pd.DataFrame()
stamp_sig_predictions = pd.read_csv(stamp_sig_predictions_path) if stamp_sig_predictions_path.exists() else pd.DataFrame()

if region_predictions.empty:
    print("WARNING: no region predictions found yet -- run Jordan's notebook first.")
region_predictions.head()


## 2. Crop OCR-relevant regions and run OCR

In [ ]:
from src.ocr import ocr_regions, crop_region

OCR_REGIONS = [
    "reference_numbers_region",
    "payment_terms_region",
    "terms_and_conditions_region",
    "total_amount_region",
    "invoice_number_region",
    "due_date_region",
    "date_region",
]

import easyocr
reader = easyocr.Reader(["en"])  # build once, reuse across images

ocr_rows = []
for doc_id, group in region_predictions[region_predictions["label"].isin(OCR_REGIONS)].groupby("document_id"):
    # TODO: img = cv2.imread(path to doc_id's image, from Rolando's manifest)
    # boxes = group.to_dict("records")
    # results = ocr_regions(img, boxes, engine="easyocr", reader=reader)
    # for r in results:
    #     ocr_rows.append({"document_id": doc_id, "region_label": r["label"], "raw_text": r["text"], "confidence": r["confidence"]})
    pass

ocr_outputs = pd.DataFrame(ocr_rows, columns=["document_id", "region_label", "raw_text", "confidence"])
ocr_outputs.head()


## 3. Check required parameters (config-driven, supports custom fields)

In [ ]:
from src.parameter_checker import check_all_fields

parameter_rows = []
for doc_id, group in ocr_outputs.groupby("document_id"):
    combined_text = " ".join(group["raw_text"].fillna(""))
    for result in check_all_fields(combined_text):
        parameter_rows.append({"document_id": doc_id, **result})

parameter_presence_results = pd.DataFrame(parameter_rows, columns=["document_id", "field_name", "required", "present", "matched_text", "match_method"])
parameter_presence_results.head()


## 4. Extract payment terms, due dates, and terms & conditions clause flags

In [ ]:
from src.terms_extraction import extract_terms_and_conditions

terms_rows = []
for doc_id, group in ocr_outputs.groupby("document_id"):
    region_texts = dict(zip(group["region_label"], group["raw_text"].fillna("")))
    extracted = extract_terms_and_conditions(region_texts)
    terms_rows.append({
        "document_id": doc_id,
        **extracted["payment_context"],
        "late_payment_flag": extracted["terms_and_conditions"]["late_payment_clause_detected"],
        "dispute_flag": extracted["terms_and_conditions"]["dispute_clause_detected"],
        "penalty_flag": extracted["terms_and_conditions"]["penalty_clause_detected"],
        "extracted_text": extracted["terms_and_conditions"]["extracted_text"],
        "summary": extracted["terms_and_conditions"]["summary"],
    })

terms_extraction_results = pd.DataFrame(terms_rows)
terms_extraction_results.head()


## 5. OCR / parameter metrics summary

In [ ]:
ocr_parameter_metrics = {
    "documents_processed": int(ocr_outputs["document_id"].nunique()) if not ocr_outputs.empty else 0,
    "mean_ocr_confidence": float(ocr_outputs["confidence"].mean()) if not ocr_outputs.empty else None,
    "required_field_presence_rate": (
        float(parameter_presence_results.loc[parameter_presence_results["required"], "present"].mean())
        if not parameter_presence_results.empty else None
    ),
}
print(json.dumps(ocr_parameter_metrics, indent=2))


In [ ]:
# --- Final export cell ---
# Save every output required by model_interface_contract.md

PATHS.predictions_dir.mkdir(parents=True, exist_ok=True)
ocr_outputs.to_csv(PATHS.predictions_dir / "ocr_outputs.csv", index=False)
parameter_presence_results.to_csv(PATHS.predictions_dir / "parameter_presence_results.csv", index=False)
terms_extraction_results.to_csv(PATHS.predictions_dir / "terms_extraction_results.csv", index=False)

PATHS.metrics_dir.mkdir(parents=True, exist_ok=True)
(PATHS.metrics_dir / "ocr_parameter_metrics.json").write_text(json.dumps(ocr_parameter_metrics, indent=2), encoding="utf-8")

member_out = PATHS.member_outputs_dir("damir_ocr_terms")
member_out.mkdir(parents=True, exist_ok=True)
ocr_outputs.to_csv(member_out / "ocr_outputs.csv", index=False)
parameter_presence_results.to_csv(member_out / "parameter_presence_results.csv", index=False)
terms_extraction_results.to_csv(member_out / "terms_extraction_results.csv", index=False)
(member_out / "ocr_parameter_metrics.json").write_text(json.dumps(ocr_parameter_metrics, indent=2), encoding="utf-8")

print('Export complete.')
